### Actionable ML Improvements

#### 1. Automate `k` Selection with Optuna
Rather than reading the elbow by eye, use **Optuna** to search for the `k` that maximises the Silhouette Score:

```python
import optuna

def objective(trial):
    k = trial.suggest_int('k', 2, 10)
    labels = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10).fit_predict(X1_scaled)
    return silhouette_score(X1_scaled, labels)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best k:", study.best_params['k'])
```

#### 2. Downstream Classification with XGBoost
Once clusters are assigned, treat them as **labels** for a supervised model — enabling prediction on new/unseen customers:

```python
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

X_cls = df[['Age', 'Annual Income', 'Spending Score']].values
y_cls = df['Cluster_Income_Spending'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

xgb = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_tr, y_tr)
print("XGBoost accuracy:", (xgb.predict(X_te) == y_te).mean())
```

#### 3. Handle Imbalanced Clusters with SMOTE
If cluster sizes are highly unequal (e.g., 80 vs 5 samples) after using clusters as downstream labels, apply **SMOTE** before training:

```python
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(random_state=42)),
    ('classifier',   XGBClassifier(random_state=42))
])
or 
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_tr, y_tr)
```

#### 4. Hyperparameter Tuning with Optuna + XGBoost
Combine both — let Optuna tune the full XGBoost pipeline:

```python
def xgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'learning_rate':   trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    model = XGBClassifier(**params, eval_metric='mlogloss', random_state=42)
    model.fit(X_tr, y_tr)
    return (model.predict(X_te) == y_te).mean()

study = optuna.create_study(direction='maximize')
study.optimize(xgb_objective, n_trials=50)
print("Best params:", study.best_params)
```

#### 5. TabTransformer / FT-Transformer for Tabular Embeddings
Transform categorical features into learnable embeddings before clustering — especially useful with high-cardinality categorical columns:
```python
# Using pytorch-tabnet or tab-transformer-pytorch
from pytorch_tabnet.tab_model import TabNetClassifier

tabnet = TabNetClassifier(n_d=8, n_a=8, n_steps=3)
tabnet.fit(X_train_num, y_train, eval_set=[(X_val_num, y_val)])
embeddings = tabnet.predict_proba(X_test)  # Use as cluster input
```


#### 6. Alternative Clustering Algorithms
- **Gaussian Mixture Models (GMM):** Soft probabilistic cluster assignment
  ```python
  from sklearn.mixture import GaussianMixture
  gmm = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
  df['GMM_Cluster'] = gmm.fit_predict(X1_scaled)
  probs = gmm.predict_proba(X1_scaled)  # Soft assignments
  ```
- **HDBSCAN:** Hierarchical DBSCAN — better at varying-density clusters
  ```python
  import hdbscan
  hdb = hdbscan.HDBSCAN(min_cluster_size=10)
  df['HDBSCAN_Cluster'] = hdb.fit_predict(X1_scaled)
  ```
#### 7. Cluster Stability Validation
Use bootstrap resampling to check if discovered clusters are stable across data subsets:
Run clustering across multiple random seeds and measure **ARI (Adjusted Rand Index)** between runs to confirm clusters are stable, not artefacts of initialisation.
```python
from sklearn.utils import resample
stability_scores = []
for _ in range(100):
    X_boot = resample(X1_scaled, random_state=None)
    km = KMeans(n_clusters=5, random_state=42).fit(X_boot)
    stability_scores.append(silhouette_score(X_boot, km.labels_))
print(f"Mean stability: {np.mean(stability_scores):.4f} ± {np.std(stability_scores):.4f}")
```

#### 8. Richer Feature Engineering
Extend the local `SpendingPropensityTransformer` to add:
- `Income_per_Age = Annual Income / Age` — earning trajectory proxy
- `Log_Income = log1p(Annual Income)` — compress right-skewed distributions
- Interaction terms: `Age × Spending Score`

#### 9. Business Deployment
- Export cluster centroids and the trained XGBoost pipeline with `joblib.dump()` for real-time scoring
- Build a **customer scoring API** (FastAPI) that returns cluster + persona label for new customers
- Feed segments into a **recommendation engine** or CRM for personalised campaigns


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import train_test_split

import scipy.cluster.hierarchy as sch
import optuna

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# =========================================================
# 1. Base Features (Income × Spending)
# =========================================================
features = ['Annual_Income', 'Spending_Score']
X1 = df[features]

scaler = StandardScaler()
X1_scaled = scaler.fit_transform(X1)

# =========================================================
# 2. Auto-select k using Optuna
# =========================================================
def kmeans_objective(trial):
    k = trial.suggest_int('k', 2, 10)
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X1_scaled)
    return silhouette_score(X1_scaled, labels)

study = optuna.create_study(direction='maximize')
study.optimize(kmeans_objective, n_trials=30)

best_k = study.best_params['k']
print("Best k (KMeans):", best_k)

# =========================================================
# 3. Train KMeans
# =========================================================
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X1_scaled)

# =========================================================
# 4. Agglomerative Clustering
# =========================================================
plt.figure(figsize=(16, 8))
linkage_matrix = sch.linkage(X1_scaled, method='ward')
sch.dendrogram(linkage_matrix)
plt.title('Dendrogram — Income × Spending')
plt.axhline(y=5, color='red', linestyle='--')
plt.show()

agg = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
df['Cluster_Agg'] = agg.fit_predict(X1_scaled)

# =========================================================
# 5. Evaluation
# =========================================================
print("\n--- Clustering Evaluation ---")

print("KMeans Silhouette:",
      silhouette_score(X1_scaled, df['Cluster_KMeans']))

print("Agglomerative Silhouette:",
      silhouette_score(X1_scaled, df['Cluster_Agg']))

print("Agglomerative DB Score:",
      davies_bouldin_score(X1_scaled, df['Cluster_Agg']))

# =========================================================
# 6. Feature Engineering (Propensity)
# =========================================================
def add_propensity(X):
    income = X[:, 0]
    spending = X[:, 1]
    propensity = spending / (income + 1e-6)
    return np.column_stack([X, propensity])

pipe = Pipeline([
    ('engineer', FunctionTransformer(add_propensity, validate=False)),
    ('scaler', StandardScaler())
])

X_prop = pipe.fit_transform(df[['Annual_Income', 'Spending_Score']].values)

df['Spending_Propensity'] = (
    df['Spending_Score'] / (df['Annual_Income'] + 1e-6)
)

# =========================================================
# 7. Visualization
# =========================================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df,
                x='Annual_Income',
                y='Spending_Score',
                hue='Cluster_KMeans',
                palette='viridis')
plt.title("KMeans Clusters")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df,
                x='Annual_Income',
                y='Spending_Score',
                hue='Cluster_Agg',
                palette='tab10')
plt.title("Agglomerative Clusters")
plt.show()

# =========================================================
# 8. Cluster Profiles
# =========================================================
profiles = df.groupby('Cluster_KMeans')[['Annual_Income', 'Spending_Score']].mean()
profiles['Size'] = df['Cluster_KMeans'].value_counts().sort_index()

print("\nCluster Profiles:\n", profiles)

# =========================================================
# 9. Downstream Classification (XGBoost)
# =========================================================
X_cls = df[['Age', 'Annual_Income', 'Spending_Score']]
y_cls = df['Cluster_KMeans']

X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Optional: handle imbalance
smote = SMOTE(random_state=42)
X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

model = XGBClassifier(n_estimators=200, eval_metric='mlogloss')
model.fit(X_tr, y_tr)

acc = (model.predict(X_te) == y_te).mean()
print("\nXGBoost Accuracy:", acc)

# =========================================================
# 10. Optuna Tuning for XGBoost
# =========================================================
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    model = XGBClassifier(**params, eval_metric='mlogloss')
    model.fit(X_tr, y_tr)

    return (model.predict(X_te) == y_te).mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=30)

print("\nBest XGBoost Params:", study_xgb.best_params)